In [8]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

NAMES = [name.strip() for name in open("names.txt", "r").readlines()]

def name_to_tensor(name):
    return torch.tensor([0]+[1 + ord(c) - ord("a") for c in name]+[0])
def tensor_to_name(tensor):
    return "".join([chr(c + ord("a")-1) for c in tensor[1:-1]])

In [ ]:
inputX = []
inputY = []

CONTEXT_WIDTH = 3
#DEVICE = "cpu"
DEVICE = "cuda"

for name in NAMES:
    t = name_to_tensor(name)
    x = [0] * CONTEXT_WIDTH

    for i in range(0, len(t)-1):
        x = x[1:] + [t[i].item()]
        inputX.append(x)
        inputY.append(t[i+1].item())

inputX = torch.tensor(inputX, device=DEVICE)
inputY = torch.tensor(inputY, device=DEVICE)

#print(inputX)
#print(inputY)        

C = torch.randn([27,2], device=DEVICE, requires_grad=True)
W1 = torch.randn([6,100], device=DEVICE, requires_grad=True)
B1 = torch.randn([100], device=DEVICE, requires_grad=True)
W2 = torch.randn([100,27], device=DEVICE, requires_grad=True)
B2 = torch.randn([27], device=DEVICE, requires_grad=True)
parameters = [C, W1, B1, W2, B2]

for iteration in range(10000):
    emb = C[inputX]
    R = emb.view(-1, 6) @ W1 + B1
    R = R @ W2 + B2
    R = F.cross_entropy(R, inputY)

    if iteration % 1 == 0:
        for param in parameters:
            param.grad = None
        R.backward()
        with torch.no_grad():
            for param in parameters:
                param -= 0.1 * param.grad
        if iteration % 100 == 0:
            print(iteration, R.item())

print("Final loss:", R.item())        

    


0 47.20932388305664
100 3.6234490871429443
200 2.861480712890625
300 3.1877286434173584
400 2.8035571575164795
500 2.9463584423065186
600 2.8565833568573
700 2.767664670944214
800 2.7115540504455566
900 2.682819128036499
Final loss: 2.6033833026885986
